# DE-DNN-IDS on Google Colab

Differential Evolution optimised DNN for intrusion detection on
CSE-CIC-IDS2018.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. On CPU the
search takes most of a day; the whole reason to use Colab is the GPU.

Run the cells in order. Sections 1-5 are setup and take about 10 minutes.
Section 6 is the experiment.

> **Colab disconnects.** Free sessions drop after roughly 90 minutes idle and
> cap out around 12 hours. Every long cell below writes its state to Google
> Drive and resumes from it, so a disconnect costs you one generation, not the
> whole run. If you get dropped: reconnect, re-run cells 1-5, then re-run the
> same search cell. It picks up where it stopped.

## 1. Confirm the GPU is actually attached

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "NO GPU - go to Runtime > Change runtime type > T4 GPU")

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__}, GPUs visible: {gpus}")
assert gpus, "No GPU visible to TensorFlow. Fix the runtime type before going on."

## 2. Get the code

In [ ]:
import os
REPO = "https://github.com/von-moyo/de-dnn-ids.git"
if not os.path.exists("/content/de-dnn-ids"):
    !git clone -q {REPO} /content/de-dnn-ids
else:
    !git -C /content/de-dnn-ids pull -q
%cd /content/de-dnn-ids
!git log --oneline -1

## 3. Dependencies

**Do not** run `pip install -r requirements.txt` here. That file pins
`numpy<2.1` for local reproducibility, and applying it in Colab downgrades the
NumPy that the pre-installed TensorFlow was built against — which forces a
runtime restart and can leave TF broken. Colab already ships everything except,
usually, `pyarrow`. Install only what is genuinely missing.

In [ ]:
import importlib

need = []
for mod, pkg in [("pyarrow", "pyarrow"), ("imblearn", "imbalanced-learn"),
                 ("seaborn", "seaborn"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)

if need:
    print("installing:", need)
    !pip install -q {" ".join(need)}
else:
    print("all dependencies already present")

import numpy, pandas, sklearn, pyarrow
print(f"numpy {numpy.__version__} | pandas {pandas.__version__} | "
      f"sklearn {sklearn.__version__} | pyarrow {pyarrow.__version__}")

## 4. Fetch the dataset

Downloaded straight from Kaggle into Colab's local disk — far faster than
uploading ~700 MB from your machine, and local disk reads much faster than
Drive during training.

**Credentials.** The clean way is Colab Secrets (key icon in the left sidebar):
add `KAGGLE_USERNAME` and `KAGGLE_KEY` from your `kaggle.json`, and enable
notebook access for both. The cell falls back to uploading `kaggle.json`
directly if the secrets are not set.

In [ ]:
import os, json

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("using Colab Secrets")
except Exception as e:
    print(f"secrets unavailable ({type(e).__name__}); upload kaggle.json")
    from google.colab import files
    up = files.upload()
    creds = json.loads(next(iter(up.values())))
    os.environ["KAGGLE_USERNAME"] = creds["username"]
    os.environ["KAGGLE_KEY"] = creds["key"]

In [ ]:
%%time
# ~700 MB. Skipped automatically if the captures are already on disk.
import glob
if len(glob.glob("data/*.parquet")) < 10:
    !kaggle datasets download -d dhoogla/csecicids2018 -p data --unzip
else:
    print("captures already present, skipping download")

In [ ]:
# Kaggle sometimes serves large members individually zipped. Extract any
# nested archives to the name the loader expects, so it reads the extracted
# copy and ignores the archive rather than loading both.
import glob, os, zipfile, shutil

for z in sorted(glob.glob("data/*.zip")):
    target = z[:-4]
    if os.path.exists(target):
        continue
    with zipfile.ZipFile(z) as zf:
        inner = [n for n in zf.namelist()
                 if n.lower().endswith((".parquet", ".csv"))]
        if len(inner) == 1:
            with zf.open(inner[0]) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 20)
            print(f"extracted {os.path.basename(target)}")

caps = sorted(glob.glob("data/*.parquet"))
print(f"\n{len(caps)} capture files:")
for c in caps:
    print(f"  {os.path.basename(c):<66} {os.path.getsize(c)/1e6:7.1f} MB")

## 5. Mount Drive for results

Colab's local disk is wiped when the session ends. Results **and the DE
checkpoint** go to Drive so a disconnect is recoverable. The dataset stays on
local disk — it is re-downloadable and Drive I/O would slow training down.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/de-dnn-ids"
STAGE1 = f"{BASE}/stage1"
FINAL = f"{BASE}/final"
CKPT = f"{BASE}/de_checkpoint.json"
for d in (STAGE1, FINAL):
    os.makedirs(d, exist_ok=True)
print(f"results  -> {BASE}\ncheckpoint -> {CKPT}")

## 6. Run

### 6a. Smoke test (~8 minutes)

Exercises the whole pipeline on a tiny sample. Confirms the data loaded, the
GPU is being used, and the class handling is right before you commit hours.
Expect: 3 classes dropped, `classes=12`, no `<-- TOO FEW` flags.

In [ ]:
%%time
!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {BASE}/smoke \
    --max_per_class 2000 --min_class_rows 200 \
    --pop_size 4 --generations 1 --fitness_epochs 3 --final_epochs 5

### 6b. Stage 1 — the DE search

110 candidate networks trained. **Time this against your own GPU** rather than
trusting an estimate: watch how long `[DE] gen 1` takes to appear, then
multiply by 10.

Re-running this cell after a disconnect resumes from the checkpoint. To start a
genuinely fresh search, delete the checkpoint file first.

In [ ]:
%%time
!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {STAGE1} --checkpoint {CKPT} \
    --max_per_class 20000 --min_class_rows 200 \
    --pop_size 10 --generations 10 --fitness_epochs 8 --seed 42

### 6c. Stage 2 — train the winner

Skips the search and trains one network on 10x the data, using the winning
configuration.

`--load_config` replays only the *hyperparameters*. The data flags are **not**
stored in it, so `--max_per_class` and `--min_class_rows` must be repeated here
or you evaluate a different class set than the one DE tuned for.

In [ ]:
%%time
!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {FINAL} \
    --max_per_class 200000 --min_class_rows 200 \
    --load_config {STAGE1}/best_config.json

## 7. Results

In [ ]:
import json, os
from IPython.display import Image, display

with open(f"{FINAL}/metrics.json") as fh:
    m = json.load(fh)

print("==== TEST METRICS ====")
for k, v in m.items():
    if k != "_sampling":
        print(f"{k:>26}: {v:.4f}")

print("\nHeadline for the writeup: macro F1 =", round(m["f1"], 4))
print("Operational false-alarm rate =",
      round(m.get("benign_false_alarm_rate", float('nan')), 4))
print("\nNOTE:", m.get("_sampling", {}).get("caveat", ""))

print("\n" + open(f"{FINAL}/classification_report.txt").read())
for fig in ("de_convergence.png", "confusion_matrix.png"):
    p = f"{STAGE1}/{fig}" if fig.startswith("de_") else f"{FINAL}/{fig}"
    if os.path.exists(p):
        display(Image(p))

---

## Reading the numbers

**Quote `benign_false_alarm_rate`, not `fpr`.** `fpr` is a macro one-vs-rest
average over all 12 classes; a dozen easy attack classes with almost no false
positives dilute it toward zero even when most benign traffic is being
misclassified. `benign_false_alarm_rate` is the fraction of genuinely benign
flows flagged as some attack, which is what the alert-fatigue argument is about.

**Accuracy and FPR are not operational estimates.** `--max_per_class` rebalances
the test set, so it does not carry real traffic's 80% benign prior. Macro
precision/recall/F1 weight classes equally and are unaffected. The caveat is
recorded in `metrics.json` under `_sampling`.

**Benign vs Infiltration is the hard case.** They are near-indistinguishable in
this dataset and the model confuses them in both directions. Expect that pair to
dominate the error budget; it is a property of CSE-CIC-IDS2018, not a bug.

**Dataset build.** This uses the deduplicated parquet redistribution, which has
6,659,532 rows against the raw release's ~16M. Deduplication collapses
FTP-BruteForce to 53 rows, DoS-SlowHTTPTest to 55 and SQL Injection to 85 —
`--min_class_rows 200` drops those three as unscoreable. Published results
almost all use the raw build, where duplicate flows appear in both train and
test and inflate the figures. **Your numbers will be lower and are not directly
comparable.** Say so explicitly. See `README.md` for the full comparison.